# Data Governance — Data Catalog (DCAT)

Registers the five data products of the **Sound Analysis & Cymatics** domain
in a machine-readable catalog, recording for each product its ownership,
storage, schema, data contract, consumers and upstream lineage, together with
a **live health**.

The catalog provides a readable JSON registry and as a **DCAT**
(Data Catalog Vocabulary) JSON-LD document, both stored in governance bucket.
Prerequisites: MinIO running; Milvus running for vector-product health (optional).

## Environment setup

In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = next(
    (p for p in [_here, *_here.parents]
     if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file()),
    None,
)
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

print("PROJECT_ROOT =", PROJECT_ROOT)

## MinIO connection — self-contained client setup

In [ ]:
from minio import Minio

MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "admin")
MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "password")
MINIO_SECURE = os.environ.get("MINIO_SECURE", "false").lower() == "true"

EXPLOITATION_BUCKET = os.environ.get("EXPLOITATION_ZONE_BUCKET", "exploitation-zone")
GOVERNANCE_BUCKET = os.environ.get("GOVERNANCE_BUCKET", "governance-zone")

CATALOG_KEY = "catalog/data_catalog.json"
CATALOG_DCAT_KEY = "catalog/data_catalog_dcat.jsonld"

DOMAIN = "Sound Analysis & Cymatics"
CATALOG_TITLE = "Cymatics Exploitation-Zone Data Catalog"

AUDIO_COLLECTION = "sound_audio_embeddings"
TEXT_COLLECTION = "sound_text_embeddings"
CYMATICS_COLLECTION = "sound_cymatics_embeddings"

DELTA_PREFIX = "metadata/observations_delta/"
MODELS_PREFIX = "models/"

def create_minio_client():
    return Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY,
                 secret_key=MINIO_SECRET_KEY, secure=MINIO_SECURE)

minio_client = create_minio_client()
print(f"MinIO connected: {MINIO_ENDPOINT}")

## Step 1: Declare the data-product registry (static metadata)

In [ ]:
DATA_PRODUCTS = [
    {
        "id": "sound_observations_delta",
        "title": "Sound observations (curated wide table)",
        "type": "structured",
        "owner": "pipeline_admin",
        "description": (
            "Single table observation row per UUID: identifiers, "
            "peak-frequency statistics, cymatics quality scores, eight Spark "
            "spectral features and the two Python features (MFCC vector, "
            "harmonic-energy ratio), plus the audio / image / video asset "
            "paths. The analytical ground truth of the domain."
        ),
        "storage": {
            "system": "Delta Lake on MinIO",
            "location": f"{EXPLOITATION_BUCKET}/{DELTA_PREFIX}",
            "format": "Delta Lake (Parquet) + CSV mirror",
        },
        "schema": "Typed schema enforced by shared/sync_delta.py",
        "data_contract": (
            "Typed and fixed schema; refreshed every 15 days via the "
            "exploitation-zone Airflow DAG; quality monitored by Great "
            "Expectations (non-null UUID, no duplicates, scores in [0,1], "
            "valid asset paths)."
        ),
        "derived_from": ["trusted-zone observations"],
        "consumers": ["KPI discovery", "Streamlit dashboard", "lineage tracker", "analysts"],
    },
    {
        "id": "sound_audio_embeddings",
        "title": "Audio embeddings (PANNs CNN14)",
        "type": "vector",
        "owner": "data_scientist",
        "description": (
            "Acoustic signature of every recording as a 2048-dim PANNs CNN14 "
            "vector, keyed by UUID, with categorical values for filtered ANN "
            "search."
        ),
        "storage": {
            "system": "Milvus (HNSW / COSINE)",
            "location": AUDIO_COLLECTION,
            "format": "FLOAT_VECTOR dim=2048",
        },
        "schema": "uuid (PK), category, source, peak_frequency_hz, "
                  "symmetry_score, pattern_stability_score, audio_embedding[2048]",
        "data_contract": (
            "Vectors L2-normalised, non-zero, of the exact declared "
            "dimensionality; idempotent upsert keyed on UUID."
        ),
        "derived_from": ["sound_observations_delta", "trusted-zone audio"],
        "consumers": ["audio classification", "audio similarity search"],
    },
    {
        "id": "sound_text_embeddings",
        "title": "Text embeddings (all-MiniLM-L6-v2)",
        "type": "vector",
        "owner": "data_scientist",
        "description": (
            "Natural-language description per observation embedded into a "
            "384-dim sentence-transformer vector for semantic metadata search."
        ),
        "storage": {
            "system": "Milvus (HNSW / COSINE)",
            "location": TEXT_COLLECTION,
            "format": "FLOAT_VECTOR dim=384",
        },
        "schema": "uuid (PK), category, source, peak_frequency_hz, "
                  "description_text, text_embedding[384]",
        "data_contract": (
            "Descriptions at most 4096 characters; vectors L2-normalised; "
            "idempotent upsert keyed on UUID."
        ),
        "derived_from": ["sound_observations_delta"],
        "consumers": ["metadata semantic search"],
    },
    {
        "id": "sound_cymatics_embeddings",
        "title": "Cymatics embeddings (CLIP ViT-B/32)",
        "type": "vector",
        "owner": "data_scientist",
        "description": (
            "Visual signature of every cymatics image as a 512-dim CLIP "
            "vector in a shared image-text space, enabling image-to-image "
            "and text-to-image queries."
        ),
        "storage": {
            "system": "Milvus (HNSW / COSINE)",
            "location": CYMATICS_COLLECTION,
            "format": "FLOAT_VECTOR dim=512",
        },
        "schema": "uuid (PK), category, source, peak_frequency_hz, "
                  "symmetry_score, pattern_stability_score, image_path, "
                  "cymatics_embedding[512]",
        "data_contract": (
            "Vectors L2-normalised; idempotent upsert keyed on UUID; refresh "
            "aligned with the exploitation-zone DAG."
        ),
        "derived_from": ["sound_observations_delta", "trusted-zone cymatics PNG"],
        "consumers": ["cymatics pattern search", "cymatics classifier training"],
    },
    {
        "id": "classifier_models",
        "title": "Classifier heads (model registry)",
        "type": "model",
        "owner": "data_scientist",
        "description": (
            "Supervised logistic-regression heads (StandardScaler -> "
            "LogisticRegression, balanced) mapping audio and cymatics "
            "embeddings to category predictions, each with a JSON model card."
        ),
        "storage": {
            "system": "MinIO object store",
            "location": f"{EXPLOITATION_BUCKET}/{MODELS_PREFIX}",
            "format": "joblib (.joblib) + metrics JSON (.metrics.json)",
        },
        "schema": "audio_classifier.joblib, cymatics_classifier.joblib "
                  "(+ matching .metrics.json model cards)",
        "data_contract": (
            "Each model carries a non-empty class list, a complete metrics "
            "report and a model_version; refreshed via the exploitation-zone "
            "DAG (every 15 days, or on demand)."
        ),
        "derived_from": ["sound_audio_embeddings", "sound_cymatics_embeddings"],
        "consumers": ["audio classification", "cymatics classification"],
    },
]

print(f"Declared {len(DATA_PRODUCTS)} data products in domain '{DOMAIN}'")

## Step 2: Probe live health across MinIO and Milvus

In [ ]:
def milvus_collection_counts():
    """Return {collection name -> entity count}, or -1 when unreachable."""
    counts = {}
    try:
        from pymilvus import MilvusClient
        MILVUS_URI = os.environ.get("MILVUS_URI", "http://localhost:19530")
        mc = MilvusClient(uri=MILVUS_URI)
        for coll in (AUDIO_COLLECTION, TEXT_COLLECTION, CYMATICS_COLLECTION):
            if not mc.has_collection(coll):
                counts[coll] = 0
                continue
            stats = mc.get_collection_stats(coll)
            counts[coll] = int(stats.get("row_count", 0))
    except Exception:
        for coll in (AUDIO_COLLECTION, TEXT_COLLECTION, CYMATICS_COLLECTION):
            counts[coll] = -1
    return counts


def object_exists(client, bucket, key):
    try:
        client.stat_object(bucket, key)
        return True
    except Exception:
        return False


def prefix_has_objects(client, bucket, prefix):
    try:
        return any(True for _ in client.list_objects(bucket, prefix=prefix, recursive=True))
    except Exception:
        return False


def probe_product_health(client, product, milvus_counts):
    """Return a live health record {status, detail} for one data product."""
    pid = product["id"]
    status, detail = "missing", ""

    if pid == "sound_observations_delta":
        present = prefix_has_objects(client, EXPLOITATION_BUCKET, DELTA_PREFIX)
        status = "available" if present else "missing"
        detail = "Delta table present" if present else "Delta table not found"

    elif pid in (AUDIO_COLLECTION, TEXT_COLLECTION, CYMATICS_COLLECTION):
        count = milvus_counts.get(pid, -1)
        if count > 0:
            status, detail = "available", f"{count} vectors"
        elif count == 0:
            status, detail = "empty", "collection exists but empty"
        else:
            status, detail = "unreachable", "Milvus not reachable"

    elif pid == "classifier_models":
        audio = object_exists(client, EXPLOITATION_BUCKET, f"{MODELS_PREFIX}audio_classifier.joblib")
        cym = object_exists(client, EXPLOITATION_BUCKET, f"{MODELS_PREFIX}cymatics_classifier.joblib")
        present = int(audio) + int(cym)
        if present == 2:
            status, detail = "available", "audio + cymatics heads present"
        elif present == 1:
            status, detail = "partial", "one head present"
        else:
            status, detail = "missing", "no model artefacts found"

    return {"status": status, "detail": detail}


milvus_counts = milvus_collection_counts()
print("Milvus collection counts:", milvus_counts)

## Step 3: Build the catalog (registry + live health)

In [ ]:
catalog_products = []
for product in DATA_PRODUCTS:
    health = probe_product_health(minio_client, product, milvus_counts)
    record = dict(product)
    record["health"] = health
    catalog_products.append(record)
    print(f"  {product['id']:<28} {health['status']:<12} {health['detail']}")

catalog = {
    "title": CATALOG_TITLE,
    "domain": DOMAIN,
    "n_products": len(catalog_products),
    "products": catalog_products,
}

## Step 4: Render the catalog as DCAT JSON-LD (W3C standard)

In [ ]:
def to_dcat_jsonld(catalog):
    """Render the catalog as a DCAT (Data Catalog Vocabulary) JSON-LD document."""
    datasets = []
    for p in catalog["products"]:
        datasets.append({
            "@type": "dcat:Dataset",
            "dct:identifier": p["id"],
            "dct:title": p["title"],
            "dct:description": p["description"],
            "dcat:theme": catalog["domain"],
            "dct:type": p["type"],
            "dct:publisher": {"@type": "foaf:Agent", "foaf:name": p["owner"]},
            "dct:conformsTo": p["data_contract"],
            "prov:wasDerivedFrom": p["derived_from"],
            "dcat:distribution": [{
                "@type": "dcat:Distribution",
                "dct:format": p["storage"]["format"],
                "dcat:accessURL": p["storage"]["location"],
                "dct:title": p["storage"]["system"],
            }],
            "cymatics:consumers": p["consumers"],
            "cymatics:status": p["health"]["status"],
        })

    return {
        "@context": {
            "dcat": "http://www.w3.org/ns/dcat#",
            "dct": "http://purl.org/dc/terms/",
            "foaf": "http://xmlns.com/foaf/0.1/",
            "prov": "http://www.w3.org/ns/prov#",
            "cymatics": "urn:cymatics:catalog#",
        },
        "@type": "dcat:Catalog",
        "dct:title": catalog["title"],
        "dct:description": (
            "Catalog of data products in the Sound Analysis & Cymatics "
            "exploitation zone."
        ),
        "dcat:theme": catalog["domain"],
        "dcat:dataset": datasets,
    }


dcat = to_dcat_jsonld(catalog)
print(f"DCAT catalog: {dcat['@type']} with {len(dcat['dcat:dataset'])} datasets")

## Display — catalog overview + per-product detail

In [ ]:
status_icons = {"available": "OK", "empty": "--", "partial": "~", "missing": "X", "unreachable": "?"}

print("=" * 64)
print(f"  Data Catalog — {catalog['domain']}")
print("-" * 64)
print(f"  Products: {catalog['n_products']}")
print("-" * 64)
for p in catalog["products"]:
    st = p["health"]["status"]
    icon = status_icons.get(st, ".")
    print(f"\n  [{icon}] {p['id']}  [{st}]")
    print(f"     Title:     {p['title']}")
    print(f"     Type:      {p['type']}")
    print(f"     Owner:     {p['owner']}")
    print(f"     Storage:   {p['storage']['system']} — {p['storage']['location']}")
    print(f"     Format:    {p['storage']['format']}")
    print(f"     Derived:   {', '.join(p['derived_from'])}")
    print(f"     Consumers: {', '.join(p['consumers'])}")
    print(f"     Status:    {p['health']['detail']}")
print("\n" + "=" * 64)

## Inspect the DCAT JSON-LD document

In [ ]:
import json
print(json.dumps(dcat, indent=2)[:2000])

## Save the catalog + DCAT JSON-LD to the governance bucket

In [ ]:
import io, json

readable = json.dumps(catalog, indent=2, default=str).encode("utf-8")
minio_client.put_object(
    GOVERNANCE_BUCKET, CATALOG_KEY,
    io.BytesIO(readable), length=len(readable),
    content_type="application/json",
)

dcat_bytes = json.dumps(dcat, indent=2, default=str).encode("utf-8")
minio_client.put_object(
    GOVERNANCE_BUCKET, CATALOG_DCAT_KEY,
    io.BytesIO(dcat_bytes), length=len(dcat_bytes),
    content_type="application/ld+json",
)

print(f"Catalog saved:      {GOVERNANCE_BUCKET}/{CATALOG_KEY} ({len(readable)/1024:.1f} KB)")
print(f"DCAT JSON-LD saved: {GOVERNANCE_BUCKET}/{CATALOG_DCAT_KEY} ({len(dcat_bytes)/1024:.1f} KB)")